# Production plan with capacity options

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ula-uab/lscm-decision-making/blob/main/cases/production_plan/notebooks/capacity_options.ipynb)

A company assembles electric bicycles in three plants and plans how many bikes each plant assembles each month from January to June. In the fifth version of the model, M4, the orders come from six retailers at the end of a distribution network in Spain, and part of the demand that a retailer does not receive in its month becomes lost sales. In the optimal plan of M4 under forecast 5, plants 1 and 2 assemble at their capacity every month. Plant 3 assembles from February, and at its capacity from March. The storage capacity of every node is full at the end of March, before the peak of the orders in April and May, and 140 orders become lost sales.

This notebook uses M4B, a version built on M4, for three questions: how far the orders can grow with every order delivered; how much a price promotion, a discount on the price of a bike in some months, must raise the orders of those months for the margin of the plan (§2) to reach the margin under the base forecast; and what closing plant 3 saves and costs. So that each change starts from a plan without lost sales, M4B starts from the base forecast, given in the first table of §1. Under it, with plant 3 open and no capacity option, every order is delivered. The base forecast is an invented demand: between 77,8 % and 81,8 % of forecast 5 in each month, and 78,9 % of its 3.550 bikes in the six months together. The orders of each month can grow over the base forecast, and the price of a month can fall with a promotion. Two capacity options, each with its cost, let the plants assemble or the central warehouse hold more than their capacity: overtime at plants 1 and 2, and rented storage next to the central warehouse. Plant 3, which has the highest unit assembly cost, 390 €/bike, has a fixed cost while it is open, and it can be closed for the six months.

Contents of the notebook:

1. Base forecast, orders of each retailer and capacity options.
2. Mixed-integer linear model with capacity options: a linear model in which some variables can only take whole values.
3. Plan of a scenario. A scenario is a set of values of the form of §3: the data that change from one plan to another (the demand growth, a price promotion, the fixed cost of plant 3 and the capacity options available with their costs) and whether plant 3 is fixed open, fixed closed, or represented by a binary variable.

The cells run in order, with ▶ or *Shift + Enter*. The form of the cell of §3 (sliders, tick boxes and a drop-down list) builds the scenario: after changing a value, run the cell again. Outside Colab the form is a set of lines such as `demand_growth = 0`: edit the value and run the cell.

The table gives the symbols that M4B adds to those of M0 to M4, which the [document of M4](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/retailers.md) lists.

| Symbol | Meaning | Term | Unit |
|---|---|---|---|
| *Data* | | | |
| $D_t$ | Orders of the six retailers together in month $t$ without demand growth | base forecast | bikes/month |
| $\sigma_r$ | Share of the orders of each month that retailer $r$ places: the shares of M4, from 10 % to 22 %, which add up to 100 % | share | — (0,14 for 14 %) |
| $\gamma_t$ | Change of the orders of every retailer in month $t$ over the base forecast, as a fraction: the value of the form in %, divided by 100. In the months of a promotion it is `growth_of_their_orders`, and in the other months `demand_growth` (fields of the form of §3); a negative value lowers the orders | demand growth | — (from −0,20 to 1,50 in the form) |
| $\pi_t$ | Price that a retailer pays for a bike ordered in month $t$: 700 €, times one minus the discount in the months of a promotion | price | €/bike |
| $\bar o_f$ | Bikes that plant $f$ can assemble in a month above its assembly capacity | overtime limit | bikes/month |
| $c^o$ | Cost of a bike assembled in overtime above the unit assembly cost of its plant | overtime cost | €/bike |
| $\bar u$ | Bikes that can be held next to the central warehouse above its storage capacity | rented storage limit | bikes |
| $c^u$ | Cost of holding one bike in rented storage for one month, above the holding cost of the central warehouse | rented storage cost | €/(bike·month) |
| $c^F_f$ | Cost of keeping plant $f$ open for one month, whatever it assembles | fixed cost | €/month |
| *Datum or variable* | | | |
| $\delta_f$ | 1 if plant $f$ is open, 0 if it is closed: $\delta_1 = \delta_2 = 1$ are data, and $\delta_3$ is a datum or a binary variable (§2) | — | — (0 or 1) |
| *Variables* | | | |
| $o_{ft}$ | Bikes that plant $f$ assembles in overtime in month $t$ | overtime | bikes/month |
| $u_t$ | Bikes held in rented storage next to the central warehouse at the end of month $t$ | rented storage | bikes |

The [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/capacity_options.md) explains all the data and results. The places and distances are real; the other data are invented.

In [ ]:
# @title Setup { display-mode: "form" }
# In Google Colab, install the example from GitHub (it takes a few seconds).
# On your own computer, install the repository environment instead (see README).
import sys
if "google.colab" in sys.modules:
    url = "git+https://github.com/ula-uab/lscm-decision-making#subdirectory=cases/production_plan"
    %pip install -q {url}
    # A session that already had the example keeps its old code unless it is reinstalled
    %pip install -q --force-reinstall --no-deps {url}
    # and unless its old code is removed from memory, so that the import below reads
    # the new one: no need to restart the session
    import importlib
    for name in [m for m in sys.modules if m == "plants" or m.startswith("plants.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
from plants import capacity_options as co

## 1. Base forecast, orders of each retailer and capacity options

The first table gives the base forecast, the orders of each retailer without demand growth and the price of each month without a promotion; the second, the data of the two capacity options and the fixed cost of plant 3. The other data are those of M4.

In [ ]:
# @title Base forecast, orders and capacity options { display-mode: "form" }
co.show_data()

## 2. Mixed-integer linear model with capacity options

Against M4, M4B adds three decisions: how many bikes each plant assembles in overtime each month, $o_{ft}$; how many bikes are held in rented storage at the end of each month, $u_t$; and, when whoever plans does not fix it, whether plant 3 is open, $\delta_3$. The orders are data computed from the base forecast and the demand growth, $d_{rt} = \sigma_r\, D_t\, (1 + \gamma_t)$. The lost-sale cost of a month is its price $\pi_t$, the same at every retailer, and takes the place of the lost-sale cost $c^l_r$ of M4. The objective adds the cost of the overtime, of the rented storage and of plant 3 while it is open.

$$
\begin{aligned}
\min \quad & \sum_{t \in T} \Big( \sum_{f \in F} \left( c_f\, p_{ft} + c^o\, o_{ft} \right) + \sum_{i \in V} c^h_i\, h_{it} + c^u\, u_t + \sum_{a \in A} c^x_a\, x_{at} + \sum_{r \in R} \left( c^b_r\, b_{rt} + \pi_t\, l_{rt} \right) \Big) + \sum_{f \in F} |T|\, c^F_f\, \delta_f \\
\text{s.t.} \quad & h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} = \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} + h_{it} && \forall i \in V,\ t \in T && \text{(stock balance at each node)} \\
& \sum_{a \in A^{\text{out}}_i} x_{at} + s_{it} \le h_{i,t-1} + p_{it} + \sum_{a \in A^{\text{in}}_i} x_{at} && \forall i \in V,\ t \in T && \text{(shipment limit at each node)} \\
& b_{r,t-1} + d_{rt} = s_{rt} + b_{rt} + l_{rt} && \forall r \in R,\ t \in T && \text{(backorder balance)} \\
& b_{rt} = \beta_r \left( b_{r,t-1} + d_{rt} - s_{rt} \right) && \forall r \in R,\ t < \tau && \text{(backorder share)} \\
& x_{at} \le K^x_a && \forall a \in A,\ t \in T && \text{(arc capacity)} \\
& \sum_{a \in A^{\text{in}}_i} x_{at} \le K^n_i && \forall i \text{ with a node capacity},\ t \in T && \text{(node capacity)} \\
& h_{it} \le K^h_i && \forall i \in V \setminus \{w\},\ t \in T && \text{(storage capacity)} \\
& h_{wt} - u_t \le K^h_w && \forall t \in T && \text{(storage capacity of the central warehouse, relaxed by the rented storage)} \\
& u_t \le \bar u && \forall t \in T && \text{(rented storage)} \\
& p_{ft} - o_{ft} \le K_{ft}\, \delta_f && \forall f \in F,\ t \in T && \text{(assembly capacity, relaxed by the overtime)} \\
& o_{ft} \le \bar o_f\, \delta_f && \forall f \in F,\ t \in T && \text{(overtime)} \\
& b_{r\tau} = 0 \quad \forall r \in R, \qquad h_{i\tau} \ge h^{\text{end}}_i \quad \forall i \in V && && \text{(backorders and required final stock at the end)} \\
& p_{ft},\ o_{ft},\ h_{it},\ u_t,\ x_{at},\ s_{rt},\ b_{rt},\ l_{rt} \ge 0, \qquad \delta_f \in \{0; 1\}
\end{aligned}
$$

with $\delta_1 = \delta_2 = 1$, $\delta_3$ a datum or a binary variable, $h_{i0}$ given, $b_{r0} = 0$, $w$ the central warehouse and $|T| = 6$ the number of months.

Two constraints of M4 are relaxed and two are new. To relax a constraint (a relaxation) is to rewrite it so that it admits plans beyond its limit: a non-negative variable lets the limit be exceeded by its value, and its cost enters the objective.

- **Assembly capacity, relaxed by the overtime.** What a plant assembles without overtime, $p_{ft} - o_{ft}$, is at most its assembly capacity. With $\delta_f = 0$ the right-hand side is 0: a closed plant assembles nothing.
- **Overtime.** An open plant assembles at most $\bar o_f$ bikes a month in overtime, and a closed plant none.
- **Storage capacity of the central warehouse, relaxed by the rented storage.** What the central warehouse holds above its storage capacity is held in rented storage, $u_t$.
- **Rented storage.** At most $\bar u$ bikes are held in rented storage.

What a plant assembles, $p_{ft}$, includes what it assembles in overtime, $o_{ft}$, and what the central warehouse holds, $h_{wt}$, includes what is held in rented storage, $u_t$. The model does not impose $o_{ft} \le p_{ft}$ or $u_t \le h_{wt}$: overtime and rented storage have a positive cost, and more of them than a plant assembles or than the warehouse holds would add cost without relaxing any limit, so a plan of least cost meets both.

**Margin.** The margin of a plan is the revenue of all the orders, delivered or not, each at the price of its month, less the cost of the plan. For given orders and prices, the revenue of the orders is the same in every plan, so the plan of least cost is the plan of highest margin. The margin is needed to compare plans whose orders or prices differ, such as a plan with a promotion and the plan under the base forecast: the cost grows with the orders, and only the margin shows whether the extra orders bring more than they cost. An order lost in the month in which it is placed adds its price to the revenue and the same amount to the cost, as a lost sale: the margin is the same as if it had not been placed (assumption 4 below gives what an order lost in a later month changes in the margin). A cost that is paid in every plan does not change which plan is optimal: it lowers every margin by the same amount. A fixed cost of plants 1 and 2, which are open in every plan (assumption 5 below), would be such a cost, and it is left out: $c^F_1 = c^F_2 = 0$. The margin of the notebook is therefore not the margin of the company, which would deduct the fixed costs of plants 1 and 2; it serves to compare plans.

**Plant 3 and $\delta_3$.** Whoever plans fixes $\delta_3$ as a datum, 1 if plant 3 is open and 0 if it is closed, or leaves it as a binary variable. Then $\delta_3$ is 1 in the optimal plan if the plan of least cost with plant 3 open, fixed cost included, costs less than the plan of least cost with plant 3 closed, and 0 if it costs more; with equal costs, both are optimal. $\delta_3$ has no month index, so plant 3 is open or closed in all six months. It has to be binary because the fixed cost is paid whole or not at all: with $\delta_3$ between 0 and 1, the constraint $p_{3t} - o_{3t} \le K_{3t}\, \delta_3$ would admit a plan with a fraction of the capacity of plant 3 for the same fraction of its fixed cost. With $\delta_3$ as a binary variable M4B is a mixed-integer linear model, and with $\delta_3$ as a datum it is a linear model.

The assumptions of M4 hold, with two changes:

- Assumption 2 of M4 says that costs are proportional to the quantities, with no fixed cost. In M4B plant 3 has a fixed cost while it is open.
- Assumption 6 of M4 says that each retailer orders the same share of forecast 5 every month. In M4B each retailer orders its share $\sigma_r$ of the base forecast raised by the demand growth.

Fractional quantities are still allowed (assumption 1 of M4), and with a demand growth the orders and the plan are in general not whole numbers. M4B adds six assumptions of its own:

1. A plan is the optimal plan for one scenario, as if its demand were known: the plans of several scenarios are compared, but no plan takes several into account.
2. The demand growth of the months of a promotion is a datum chosen by whoever plans: M4B does not represent how the orders respond to the discount. The promotion takes no orders from the other months.
3. The discounted price applies to every order of the months of the promotion, also to the orders that would have been placed without it, and the promotion has no other cost, such as advertising.
4. An order backordered in one month and lost in a later month costs the price of the later month: the model counts the backorders of a retailer together, without the month in which each was placed. If the two prices differ, the order adds the price of its month to the revenue and the price of the later month to the cost, besides the backorder cost of each month that it waits. Compared with an order not placed, an order placed in a month of a promotion and lost in a month without it therefore lowers the margin by the difference between the two prices plus its backorder cost. An order placed in a month without promotion and lost in a month of a promotion raises the margin by that difference and lowers it by its backorder cost. This does not happen in the plans at the growths that answer the questions of the promotion in §3: in them every lost sale is lost in the month in which it is placed. In the tables of the cell, it is enough to check that no retailer has lost sales in the month that follows a month in which it has backorders.
5. Plants 1 and 2 are open in all six months.
6. Overtime is available only at plants 1 and 2, $\bar o_3 = 0$, and rented storage only next to the central warehouse.

## 3. Plan of a scenario

The cell solves M4B for the scenario of the form and gives:

- the cost and the margin of the plan of the scenario;
- the margin under the base forecast, to compare with the margin of the scenario. It is the margin of the plan with a demand growth of 0 and no promotion, and the same setting of plant 3, the same fixed cost and the same capacity options as the scenario;
- the capacity utilization of each plant in the six months: what the plant assembles in the six months, overtime included, as a share of its assembly capacity of the six months. With overtime it can exceed 100 %; for a closed plant it is 0 %;
- the price and the orders of each month;
- the bikes assembled at each plant in each month, with the part assembled in overtime;
- the stock at the end of each month at each stage, with the part held in rented storage. The table gives the stock of four stages of the network: the plants, the central warehouse, the regional warehouses and the local warehouses. The rail terminal, a node between the central warehouse and regional warehouse 2, and the retailers, the stage at the end of the network, hold no stock and are not in the table. The stock at the end of a month is held for the orders of later months, the seasonal stock, or, at the end of June, as the required final stock;
- if there are lost sales, the lost sales of each retailer in each month, and its backorders at the end of each month.

The cell also says whether another plan has the same optimal cost: if one has, the tables show one of the optimal plans, and in another one the plants that assemble or the retailers with lost sales may differ. The figure has two panels. On the left, the capacity utilization of each plant in each month: what the plant assembles in the month, overtime included, divided by its assembly capacity of the month, in %, with a dashed line at 100 %, the assembly capacity. On the right, the lost sales of each retailer in each month, in bikes.

**Demand growth.** `demand_growth`, in %, raises the orders of every retailer in every month by the same percentage over the base forecast; a negative value lowers them. With plant 3 open and no capacity option, raise it from 0 to 40 %: from which growth on are there lost sales, and what is the capacity utilization of plant 3 in the six months with the first growth that has lost sales?

**Price promotion.** The tick boxes choose the months of the promotion. `discount`, in %, lowers the price of those months, and `growth_of_their_orders`, in %, is the demand growth of those months: in them it takes the place of `demand_growth`. With plant 3 open, no capacity option and `demand_growth` at 0, put a discount of 5 % in April and May: how much must the promotion raise the orders of those months for the margin of the scenario to reach the margin under the base forecast? Raise `growth_of_their_orders` one point at a time: the answer is the first value with which the margin of the scenario is equal to or above the margin under the base forecast; if no value up to 150 % reaches it, the answer is that the promotion does not reach it. Then untick April and May, tick January, February and March, and do the same. Repeat both with a discount of 7,5 %: with that discount, do April and May need a larger growth than January to March, and why? With a discount of 7,5 % in April and May and the growth found, which retailers have lost sales, and in which month?

**Plant 3.** With `plant_3` set to `open` or `closed`, plant 3 is open or closed for the six months, as a datum; with `variable`, $\delta_3$ is a binary variable (§2). `fixed_cost_of_plant_3`, in €/month, is paid in each of the six months if plant 3 is open. Its default is 0 €/month, so that with plant 3 open as a datum the cost of the plan leaves the fixed cost out, which does not change which plan is optimal (§2). The case gives 20.000 €/month (second table of §1). Set that value to compare a plan with plant 3 open with a plan with plant 3 closed. Set it also with `variable`, because then the value of $\delta_3$ in the optimal plan depends on the fixed cost. Close plant 3 under the base forecast (`demand_growth` at 0): which retailers have lost sales, and why those? Is the margin of that plan above the margin of the plan with plant 3 open and a fixed cost of 20.000 €/month? The answer takes two runs under the base forecast: one with `closed`, and one with `open` and `fixed_cost_of_plant_3` at 20.000; in each, read the margin of the plan of the scenario. With `variable` and a fixed cost of 20.000 €/month, is plant 3 open or closed in the optimal plan under the base forecast, and from which demand growth on, raising it from 0, does that change?

**Overtime and rented storage.** What closing plant 3 costs depends on what can replace its capacity. `overtime` lets plants 1 and 2 assemble above their assembly capacity, up to the overtime limits of the second table of §1, at `overtime_cost`, in €/bike above the unit assembly cost of the plant; unticked, the overtime limits are 0. `rented_storage` lets the central warehouse hold up to 200 bikes above its storage capacity, at `rented_storage_cost`, in €/(bike·month) above its holding cost; unticked, the rented storage limit is 0. The default costs are those of that table. With plant 3 closed and `demand_growth` at 0, compare the two options one at a time and both together. With which option, if any, is every order delivered? Compared with plant 3 closed and no option, which option lowers the lost sales least, and which lowers the cost least? How many lost sales and how much cost do both together save compared with overtime alone? With both options, is the margin with plant 3 closed above the margin with plant 3 open, also with both options and with a fixed cost of 20.000 €/month?

In [ ]:
# @title Plan of a scenario { display-mode: "form" }
# Demand growth over the base forecast, in every month without promotion (%)
demand_growth = 0  # @param {type:"slider", min:-20, max:50, step:1}
# Months of the price promotion
january = False  # @param {type:"boolean"}
february = False  # @param {type:"boolean"}
march = False  # @param {type:"boolean"}
april = False  # @param {type:"boolean"}
may = False  # @param {type:"boolean"}
june = False  # @param {type:"boolean"}
# Discount on the price of those months (%) and growth of their orders (%)
discount = 0  # @param {type:"slider", min:0, max:30, step:0.5}
growth_of_their_orders = 0  # @param {type:"slider", min:0, max:150, step:1}
# Plant 3 open, closed or a binary variable, and what it costs to keep it open (EUR/month)
plant_3 = "open"  # @param ["open", "closed", "variable"]
fixed_cost_of_plant_3 = 0  # @param {type:"slider", min:0, max:50000, step:1000}
# Overtime at plants 1 and 2 (EUR/bike) and rented storage (EUR/(bike*month))
overtime = False  # @param {type:"boolean"}
overtime_cost = 100  # @param {type:"slider", min:10, max:200, step:10}
rented_storage = False  # @param {type:"boolean"}
rented_storage_cost = 12  # @param {type:"slider", min:1, max:40, step:1}
co.plan(growth=demand_growth,
        promotion_months=co.chosen_months(january, february, march, april, may, june),
        discount=discount, promotion_growth=growth_of_their_orders,
        plant_3=plant_3, fixed_cost=fixed_cost_of_plant_3,
        overtime=overtime, overtime_cost=overtime_cost,
        rented_storage=rented_storage, rented_cost=rented_storage_cost);

The script `capacity_options_solver.py`, in the folder of the case, prints every table of the [document of this version](https://github.com/ula-uab/lscm-decision-making/blob/main/cases/production_plan/capacity_options.md) except the notation. It also prints the numbers that the document gives in its text and not in its tables, such as the largest demand growth with which every order is delivered.